# 10 · Training on self-gravitating data — frozen vs fine-tuned vs fresh

Jason suggested training on the self-gravitating data. The pairs he shipped can't do it:
their dirty cubes differ from clean by 0.4–7% RMS where the line-emission training set differs
by ~50%, so a network trained on them learns the identity (measured 2026-09-04, PROGRESS.md).
So the corruption was applied here instead, `dirty = beam ⊛ clean + beam ⊛ noise`, with the
beam recovered from the v2 pair. Five disks, four landing in the training set's difficulty
band.

**Three arms, one question each:**

| arm | init | trains on SG | answers |
|---|---|---|---|
| `frozen` | `winner_aug` seed 43 | no | how bad is the domain gap, on data with a *known* operator |
| `finetune` | `winner_aug` seed 43 | yes | does SG training close it |
| `fresh` | random | yes | does line-emission pretraining help or hurt |

`frozen` is the baseline everything is measured against. It is known to fail badly on real SG
data (M0 −86.5%), but that was with an estimated operator on a differently-generated cube.

**Scoring is on the holdout cube only**, which is never trained or validated on. Reported as
moment improvement (M0/M1/M2), the metric that matters per RULES.md #4, not PSNR alone.

## 0. Bootstrap

In [ ]:
import os, sys, subprocess, glob

ON_KAGGLE = os.path.exists('/kaggle')
BRANCH = 'midterm-prep'
if ON_KAGGLE:
    REPO = '/kaggle/working/EXXA'; PKG = os.path.join(REPO, 'DENOISING_DIFFUSION')
    if not os.path.exists(REPO):
        subprocess.run(['git','clone','--branch',BRANCH,'--depth','1',
                        'https://github.com/KrishanYadav333/EXXA.git',REPO], check=True)
    else:
        subprocess.run(['git','-C',REPO,'fetch','origin',BRANCH], check=True)
        subprocess.run(['git','-C',REPO,'reset','--hard','origin/'+BRANCH], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps',
                    'pytorch-msssim','bettermoments'], check=True)
    os.chdir(os.path.join(PKG,'notebooks')); sys.path.insert(0, PKG)

    # The synthesized SG pairs live in folders named run_9<xxx>_..., which is how they are
    # told apart from the line-emission set that uses the same run_<id>_<step>_rt_<pp> shape.
    hits = [p for p in glob.glob('/kaggle/input/**/run_9*_rt_*', recursive=True)
            if os.path.isdir(p)]
    if not hits:
        raise FileNotFoundError('No run_9*_rt_* folders under /kaggle/input -- attach the sg_synth Dataset.')
    DATA_DIR = os.path.dirname(hits[0])

    ck = glob.glob('/kaggle/input/**/winner_aug_seed43.*', recursive=True)
    if not ck:
        raise FileNotFoundError('winner_aug_seed43 checkpoint not found under /kaggle/input.')
    WINNER_CKPT = ck[0]
else:
    if os.path.basename(os.getcwd()) != 'notebooks' and os.path.isdir('notebooks'):
        os.chdir('notebooks')
    sys.path.insert(0, os.path.abspath('..'))
    DATA_DIR = '../self-gravitating cube and dirty cube/sg_synth'
    WINNER_CKPT = '../models/08-seeds/winner_aug_seed43.pth'

print('DATA_DIR:   ', DATA_DIR)
print('WINNER_CKPT:', WINNER_CKPT)

## 0b. Pull latest `src/` (re-run anytime, no restart)

Updates the library only, never these cells (RULES.md #2).

In [ ]:
if ON_KAGGLE:
    subprocess.run(['git', '-C', REPO, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', 'FETCH_HEAD'], check=True)
    print(subprocess.run(['git','-C',REPO,'log','--oneline','-1'],
                         capture_output=True, text=True).stdout)
    import importlib, src; importlib.reload(src)

## 1. Config

In [ ]:
import time, json, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.cube_split import split_cubes
from src.data.fits_cube_dataset import FITSChannelDataset
from src.training.sweep import train_unet
from src.evaluation.moment_maps import generate_moment_maps, signal_mask, moment_improvement

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

TARGET_SIZE = 256
N_SAMPLES   = 120          # channels sampled per cube; only 3 training disks, so sample more
NW          = 2 if torch.cuda.is_available() else 0

# winner_aug's architecture and optimiser, so `frozen`/`finetune`/`fresh` differ only in
# initialisation and whether they train. Any other difference would confound the comparison.
WINNER = dict(base_channels=48, channel_multipliers=(1, 2, 4, 8),
              lr=8.196504330730313e-4, alpha=0.8877681051398497,
              sched_patience=8, batch_size=8)

# Fine-tuning wants a gentler LR than training from scratch, otherwise the first few steps
# destroy the pretrained weights and `finetune` collapses into an expensive `fresh`.
FINETUNE_LR_SCALE = 0.1

CKPT_DIR = '../results/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'{device} | target {TARGET_SIZE}px | {N_SAMPLES} samples/cube')

## 2. Data — leakage-safe cube-level split

In [ ]:
train_cubes, val_cubes, holdout_cubes = split_cubes(
    data_dir=DATA_DIR, n_holdout=1, val_fraction=0.25, seed=SEED)

_kw = dict(n_samples=N_SAMPLES, target_size=TARGET_SIZE, seed=SEED,
           subtract_continuum=False, verbose=False)
train_ds = FITSChannelDataset(train_cubes, **_kw)
val_ds   = FITSChannelDataset(val_cubes, **_kw)

d, c = train_ds[0]
print(f'train {len(train_ds)} items / {len(train_cubes)} cubes | '
      f'val {len(val_ds)} / {len(val_cubes)} | holdout {len(holdout_cubes)}')
print(f'item: dirty {tuple(d.shape)} clean {tuple(c.shape)}')
assert d.shape == c.shape

# The identity baseline on this data. If a trained arm cannot beat this, it has learned
# nothing -- and on the pairs Jason shipped, identity would already have been near-perfect.
_mse = np.mean([float(torch.mean((train_ds[i][0] - train_ds[i][1])**2))
                for i in range(min(24, len(train_ds)))])
print(f'identity baseline MSE on train: {_mse:.5f}')

## 3. Load `winner_aug`, the starting point for two of the three arms

In [ ]:
_ck = torch.load(WINNER_CKPT, map_location='cpu', weights_only=False)
WINNER_SD = _ck['model_state_dict']
print(f"winner_aug: epoch {_ck['epoch']}, val_loss {_ck['val_loss']:.6f}, "
      f"base {_ck['base_channels']}, mults {_ck['channel_multipliers']}")
assert _ck['base_channels'] == WINNER['base_channels'], 'architecture mismatch with WINNER config'
assert tuple(_ck['channel_multipliers']) == WINNER['channel_multipliers']

## 4. Train the arms

`frozen` does not train, it only gets scored. Checkpoints are persisted the moment each arm
finishes (RULES.md #1) rather than in a cleanup cell.

In [ ]:
ARMS = [
    dict(name='finetune', init=WINNER_SD, lr_scale=FINETUNE_LR_SCALE),
    dict(name='fresh',    init=None,      lr_scale=1.0),
]

results = {}
for arm in ARMS:
    name = arm['name']
    cfg = dict(WINNER)
    cfg['lr'] = cfg['lr'] * arm['lr_scale']
    print(f"\n{'='*70}\n{name}  (lr {cfg['lr']:.2e}, init {'winner_aug' if arm['init'] else 'random'})\n{'='*70}")
    t0 = time.time()
    out = train_unet(train_ds, val_ds, device,
                     init_state_dict=arm['init'],
                     min_epochs=10, max_epochs=60, patience=8,
                     num_workers=NW, seed=SEED,
                     ckpt_path=os.path.join(CKPT_DIR, f'sg_{name}.pth'),
                     verbose=True, **cfg)
    out['minutes'] = (time.time() - t0) / 60
    results[name] = out
    print(f"  {name}: PSNR {out['psnr']:.3f} dB, SSIM {out['ssim']:.5f}, "
          f"best epoch {out['best_epoch']}, {out['minutes']:.1f} min")

    # persist immediately, next to the checkpoint train_unet already wrote
    with open(os.path.join(CKPT_DIR, f'sg_{name}_metrics.json'), 'w') as f:
        json.dump({k: v for k, v in out.items() if not hasattr(v, '__len__') or isinstance(v, (str, list))}, f, indent=2, default=str)

## 5. Score every arm on the holdout cube

Moment improvement over the dirty cube, signal-masked, which is the metric the project
actually reports (RULES.md #4). `frozen` is scored here without ever having trained.

In [ ]:
import math
from astropy.io import fits
import torch.nn.functional as Fn
from src.models.unet import UNet

hold = holdout_cubes[0]
print('holdout:', os.path.basename(hold['folder']))

with fits.open(hold['clean'], memmap=True) as h:
    hdr = h[0].header; clean_cube = h[0].data[:]
with fits.open(hold['dirty'], memmap=True) as h:
    dirty_cube = h[0].data[:]
velax = (hdr['CRVAL3'] + (np.arange(clean_cube.shape[0]) + 1 - hdr['CRPIX3']) * hdr['CDELT3']) * 1000.0
print(f'  cube {clean_cube.shape}, dv {hdr["CDELT3"]:.4f} km/s')


def denoise_cube(state_dict, cube):
    net = UNet(in_channels=1, out_channels=1,
               base_channels=WINNER['base_channels'],
               channel_multipliers=WINNER['channel_multipliers'],
               time_emb_dim=128, num_res_blocks=2,
               groups=math.gcd(8, WINNER['base_channels']), beam_dim=0).to(device)
    net.load_state_dict(state_dict); net.eval()
    C, H, W = cube.shape
    los = cube.reshape(C, -1).min(axis=1); his = cube.reshape(C, -1).max(axis=1)
    rng = his - los
    out = np.empty_like(cube, dtype=np.float32)
    with torch.no_grad():
        for s in range(0, C, 8):
            blk = cube[s:s+8].astype(np.float64)
            lo, hi = los[s:s+8], his[s:s+8]
            den = np.where((hi - lo) > 0, hi - lo, 1)[:, None, None]
            t = torch.from_numpy((blk - lo[:, None, None]) / den)[:, None].float().to(device)
            t = Fn.interpolate(t, (TARGET_SIZE, TARGET_SIZE), mode='bilinear', align_corners=False)
            p = net(t, torch.zeros(t.size(0), dtype=torch.long, device=device), None)
            b = Fn.interpolate(p, (H, W), mode='bilinear', align_corners=False)[:, 0].cpu().numpy()
            for k in range(b.shape[0]):
                out[s+k] = b[k] * rng[s+k] + los[s+k] if rng[s+k] > 0 else los[s+k]
    return out


m_clean = generate_moment_maps('', data_velax=(clean_cube.astype(np.float64), velax))
m_dirty = generate_moment_maps('', data_velax=(dirty_cube.astype(np.float64), velax))

STATES = {'frozen': WINNER_SD}
for name in results:
    _c = torch.load(os.path.join(CKPT_DIR, f'sg_{name}.pth'), map_location='cpu', weights_only=False)
    STATES[name] = _c['model_state_dict']

scores = {}
for name, sd in STATES.items():
    t0 = time.time()
    den = denoise_cube(sd, dirty_cube)
    m_den = generate_moment_maps('', data_velax=(den.astype(np.float64), velax))
    imp = moment_improvement(m_clean, m_dirty, m_den)
    scores[name] = imp
    print(f"  {name:9s} M0 {imp['M0']:+7.1f}%  M1 {imp['M1']:+7.1f}%  M2 {imp['M2']:+7.1f}%   "
          f"({(time.time()-t0)/60:.1f} min)")

## 6. Results

In [ ]:
# `frozen` never goes through train_unet, so it has no fixed-metric evaluation of its own.
# Scoring it here with val_metrics -- the SAME function, on the SAME val split -- rather than
# printing nan, which put the hole exactly where the baseline belongs. A baseline measured a
# different way would not be comparable (RULES.md #4).
from torch.utils.data import DataLoader
from src.training.sweep import val_metrics
from src.training.architectures import build_model

_val_loader = DataLoader(val_ds, batch_size=8, shuffle=False)
pix = {}
for _name, _sd in STATES.items():
    _net = build_model('unet', base_channels=WINNER['base_channels'],
                       channel_multipliers=WINNER['channel_multipliers'],
                       use_beam=False, n_neighbors=0, out_channels=1, latent_dim=128).to(device)
    _miss, _unexp = _net.load_state_dict(_sd, strict=False)
    assert not _miss and not _unexp, f'{_name}: state dict mismatch'
    pix[_name] = val_metrics(_net, _val_loader, device, use_beam=False, arch='unet')

print(f"{'arm':10s} {'PSNR':>8} {'SSIM':>9} {'M0':>9} {'M1':>9} {'M2':>9}")
for name in ['frozen', 'finetune', 'fresh']:
    if name not in scores:
        continue
    s = scores[name]
    psnr, ssim = pix[name]['psnr'], pix[name]['ssim']
    print(f"{name:10s} {psnr:8.3f} {ssim:9.5f} {s['M0']:+9.1f} {s['M1']:+9.1f} {s['M2']:+9.1f}")

summary = {n: dict(moments=scores[n],
                   psnr=pix[n]['psnr'], ssim=pix[n]['ssim'], mse=pix[n]['mse'],
                   minutes=(None if n == 'frozen' else results[n]['minutes']))
           for n in scores}
os.makedirs('../results/self-gravitating', exist_ok=True)
with open('../results/self-gravitating/sg_training_arms.json', 'w') as f:
    json.dump(dict(holdout=os.path.basename(hold['folder']),
                   n_train_cubes=len(train_cubes), arms=summary), f, indent=2, default=str)
print('\nsaved -> results/self-gravitating/sg_training_arms.json')
print('checkpoints in', CKPT_DIR, '-- download these, they are not in the notebook Output by default')